# Project K, Series Three step 5: the property probes, Qwen2.5-Omni-3B

**What this notebook is.** The Handover Correction's last
step (Nov 3, 2026). Round 2's presence helpers saturated:
1.00 at every state, no dynamic range, and a silence class
that leaked across its split. This notebook asks a finer
question with a harder instrument. The sounds are tones
that vary on two factors at once: pitch (500 vs 1500 Hz)
and onset (0.5 vs 2.5 s), 90 unique instances per cell,
every waveform a distinct identity, split by instance
inside each cell. The helpers now read PROPERTIES (high vs
low pitch; late vs early onset), at two positions (the
sound slots pooled, and the final answer position), with
200-permutation nulls in the focus band (states 16 to 25,
decoder layers 15 to 24). A presence helper rides along as
the sanity check, with round 2's 5-shuffle controls. If a
property reads well above its null in the band where the
surgery found the handover, the property survives in
readable form there; if it sits at its null, presence was
all that ever survived. Everything below about round 2 is
the machinery's origin, kept as context; this paragraph
governs what runs.

**Why this notebook exists.** Round 1 of the tracing walked
the sound through Qwen 3B's layers with per-layer helpers
(tiny classifiers trained on one layer's numbers at a time)
and an attention watch. The run finished clean, and then
its own control voided it: a helper trained on shuffled
labels, where there is nothing real to learn, sometimes
scored perfectly too. With 64 training clips against a
2,048-number layer vector, the helpers could memorize
shapes instead of reading the sound. Round 1 also left a
question open: its sanity gate found only 1 audio token in
a 287-position row, where this model family normally
spreads audio across many positions. Round 2 fixes the
probe design. The method, the question, and the registered
predictions are unchanged from the tracing plan.

**The goal.** Find the layer where the loud tone fades
inside the model, and test whether the speech is ever lost
at all. Round 2 carries the helpers and the attention
watch. Surgery (activation patching) is still the follow-up
build, and still the causal check on whatever this round
finds.

**What changed, exactly three things.**

1. **The token check comes first.** Before any extraction,
   one tone clip and one speech clip go through the
   processor, and the runner prints the full histogram of
   token ids in each processed row: every id that occurs,
   its count, and its decoded name. The positions the
   notebook pools over are derived from what the histogram
   shows: the audio candidate id that occurs at
   audio-frame scale, not the first candidate that merely
   occurs once (that is what round 1's gate settled for).
   If the derived audio-position count is under 10 for a
   clip, the step prints TOKEN CHECK FAILED and stops the
   run before anything is extracted. The transcript is
   saved as `property_token_check.txt`.
2. **A corrected battery.** 420 clips: 360 factorial tones, 60
   class (tone, speech, silence), each 4.0 seconds,
   generated inside the runner with a fixed seed and fed
   as raw WAV, no compression anywhere. Inside each class
   the clips vary, so a helper must read the sound rather
   than memorize one waveform. Every clip is measured and
   printed before use: per-second RMS and a waveform hash,
   the feed proof.
3. **A smaller, regularized readout.** Per layer, the
   vectors at the verified audio positions are averaged.
   Two helpers read each layer: a logistic regression with
   strong regularization (C=0.01) on standardized features,
   and the same probe after the features are reduced to 64
   principal components. Each helper trains on 240 clips
   per class and is tested on 120 held-out clips per
   class, the same split at every layer. Beside every real
   accuracy, the results carry the shuffled-label control
   as the mean of 5 shuffles trained and tested
   identically.

**The helpers, in plain words.** A helper is a tiny
classifier we train ourselves, on one layer's numbers only.
We play clips we chose, so we always know which clips carry
the sound and which do not. For each layer, the helper
learns the shape that layer's numbers take when the sound
is in them, then walks the layers on clips it never saw in
training. The layer where the helper stops finding the
sound is the neighborhood where the sound faded. The helper
only reads; it never changes the model, and the model is
never taught anything.

**Readout rules, fixed in advance** (from the tracing
plan, unchanged):

- A helper failing at a layer means "not readable by this
  helper," which can also mean the information changed
  shape. No stage is declared lost on the helpers alone;
  the surgery build is the causal check.
- The shuffled-label control must sit at chance (about
  0.50). A control that beats chance means the helper is
  cheating on some side pattern, and the run is void until
  fixed.
- Predictions, registered before round 1 and unchanged:
  the tone stays readable into the middle layers, then
  fades; the attention the answer pays to the tone's
  tokens is near zero; the speech stays readable at every
  layer, because its transcription is perfect, so its
  denial lives in how the answer is formed, past the
  information.

**Where it runs.** On the free Colab account (kadjei1),
under the routing rule: the 3B is a small model. **How to
run it.** Run the cells from the top, in order. Step 1 sets
up and proves the stack. Step 2 gets the two master sounds.
Step 3 writes the runner and runs its check mode (proof,
loading candidates, the token check). Step 4 runs battery
mode: clip generation, extraction, the helpers, the
attention readout, and a plain-language tail.

**The one rule.** We claim only what we ran: these sounds,
this model, these layers, this run.

## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Tracing addition.** At the end, the cell proves the
stack it just installed: it prints the imported
transformers version and the file it loaded from, and
stops the run if the version is not the pinned 4.52.4.
It also proves scikit-learn, the library the helper
classifiers are built with: if it is missing, the cell
installs it, and either way it prints the version it
found. This notebook's model runs under the session's
own Python, so the versions printed here are the run's
versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Tracing addition: prove the installed stack --------------------------
# The model and the probes run under this same session
# Python, so the versions installed above are the versions
# the run uses. Prove them here, and stop if the
# transformers pin is not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")

# The helper classifiers are scikit-learn logistic
# regressions. Colab sessions normally ship scikit-learn;
# install it only if it is missing, and print the version
# either way so the run record names what trained the
# helpers.
try:
    import sklearn
    print("scikit-learn version:", sklearn.__version__)
except ImportError:
    print("scikit-learn missing; installing scikit-learn==1.6.1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn==1.6.1"], check=False)
    import sklearn
    print("scikit-learn version:", sklearn.__version__)


## Step 2 · Get the two master sounds

Two files from your Google Drive folder named
`codec-control` anchor this run:

- `codec_tone_ref_raw.wav`: the loud 1000 Hz tone master,
  uncompressed. The token check's tone clip.
- `codec_speech_raw.wav`: the probe sentence master, the
  swap study's own speech track, uncompressed. The token
  check's speech clip, and the source every speech clip in
  the battery is rendered from.

The next cell links your Drive, finds that folder, and
copies both files to a plain work folder here. The runner
re-measures everything from inside the session before it
believes any of it.

Do not go on until both files say "ready". If a file is
missing, the cell stops and names it.

In [ ]:
# --- Get the two master sounds ---------------------------------------------------
import os, shutil, subprocess

FILES = ["codec_tone_ref_raw.wav", "codec_speech_raw.wav"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/codec-control"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/codec-control"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

missing = []
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.2f} MB{extra})")
    else:
        missing.append(name)
        print("MISSING:", name)
if missing:
    print()
    print("!" * 72)
    print("STOP. These master files are missing:")
    for name in missing:
        print("   ", name)
    print("Check the Drive folder name is exactly codec-control, or")
    print("upload the files by hand, then run this cell again.")
    print("Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: master files missing in Step 2")
print()
print("ready: both masters (tone, speech)")

## Step 3 · Write the runner, then check: proof, candidates, token check

The model never runs inside this notebook's cells directly.
It runs in a plain script, `/content/qwen_property_run.py`,
under this session's own Python, the Qwen notebooks'
pattern. The first cell below writes that script. The
second runs it in **check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner
   imported, with the transformers pin checked a second
   time: exactly 4.52.4, or the run stops. The scikit-learn
   version prints here too, and one note explains the single
   deliberate difference from the earlier runners' load
   call: the candidates load with eager attention, because
   this build must return attention weights and the default
   sdpa kernel does not.
2. **The loading candidates.** The codec runner's list: the
   3B in 16-bit form (float16) first, the 7B AWQ build
   second, only if the small one fails. The talker is turned
   off on whichever loads. A failed candidate prints its
   full traceback and appends the same words to
   `property_load_log.txt` in the tracing folder.
3. **The token check.** One tone clip and one speech clip
   (the masters from Step 2) go through the processor. For
   each, the runner prints the full histogram of token ids
   in the processed row (every id, its count, its decoded
   name, special tokens flagged), the processor's audio
   feature lengths, and every candidate audio id with its
   count. The pooling id is derived from the histogram: the
   candidate that occurs at audio-frame scale. Both clips
   must derive the same id, and its count must be at least
   10, or the step prints TOKEN CHECK FAILED and stops the
   run (exit code 11) before any extraction. One forward
   pass on the tone clip then confirms the hidden states
   come back and fixes the layer count and hidden size.
   The whole transcript is saved as
   `property_token_check.txt`. No clips are generated and
   no extraction runs in check mode.

In [ ]:
# --- Write the tracing round 2 runner script -------------------------------------
# The runner holds the proof, the loading candidates, the
# token check, the clip generation, the extraction, the
# helpers, and the attention readout. It is written to a
# plain file here and only ever run under this session's
# own Python, the interpreter Step 1 installed into.
RUNNER_SOURCE = '\n# qwen_property_run.py - the tracing round 2 runner for\n# Qwen2.5-Omni-3B. Written by\n# project_k_qwen_property_colab.ipynb. It runs under the\n# notebook session\'s own Python: the Qwen notebooks install\n# their stack into the session (Step 1), and there is no\n# private environment to run under. Modes: "check" runs the\n# proof, the loading candidates, and the TOKEN CHECK, then\n# stops. "battery" does all of that and then generates the\n# probe clips, extracts the per-layer features, trains the\n# helpers (two readouts, 5-shuffle controls), reads the\n# attention subset, and saves every output.\n#\n# Exit codes, one per failure family, all loud:\n#   3  the stack proof failed (transformers is not 4.52.4)\n#   4  both loading candidates failed\n#   7  a feed returned no audio (never padded, never scored)\n#   9  probe clip generation failed its own measurements\n#   10 the extraction count does not match the manifest count\n#   11 the token check failed: the audio positions could\n#      not be settled from the histogram, or their count is\n#      implausible, or the thinker returned no hidden states\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nimport sklearn\nprint("scikit-learn version:", sklearn.__version__)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\nprint("attention implementation note: the loading candidates below pass")\nprint("attn_implementation=\'eager\'. The attention readout is a required")\nprint("output of this build, and the default sdpa kernel does not return")\nprint("attention weights. The sequences here are short, so the memory")\nprint("cost is small, and the hidden-state math is the same.")\n\n# --- Homes for the outputs -------------------------------------------------------\n# The tracing outputs live in a Drive folder named\n# tracing-qwen3b, created here if it is missing, with /content\n# as the second home, the house pattern.\nDRIVE_TRACING = "/content/drive/MyDrive/tracing-qwen3b"\nLOCAL_TRACING = "/content/tracing-qwen3b"\nfor d in (DRIVE_TRACING, LOCAL_TRACING):\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\ndef out_paths(name):\n    return [os.path.join(DRIVE_TRACING, name),\n            os.path.join(LOCAL_TRACING, name)]\n\ndef save_both(name, write_fn):\n    saved = []\n    for p in out_paths(name):\n        try:\n            write_fn(p)\n            saved.append(p)\n            print("saved:", p)\n        except Exception as e:\n            print("could not save", p, "-", str(e)[:160])\n    return saved\n\n# --- The load log ------------------------------------------------------------------\nimport traceback, datetime\n\nLOAD_LOG_PATHS = out_paths("property_load_log.txt")\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B tracing round 2 load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the codec runner\'s candidates ---------------------------------\n# Same candidates, same order, same talker-off call as\n# round 1. The one printed difference is\n# attn_implementation="eager" (see the proof lines above\n# for why).\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto",\n            attn_implementation="eager")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\nTHINKER = model.thinker\nprint("thinker:", type(THINKER).__name__)\n\n# --- The feed path -----------------------------------------------------------------\n# Sound alone, the codec control\'s raw arm form: the\n# conversation carries an audio element naming the WAV file\n# and a text element, and no image and no video element.\n# process_mm_info decodes the audio element with the same\n# librosa call at 16 kHz mono it uses for a video\n# soundtrack. The text is the Series Two presence question\n# for the clip\'s sound, fixed per class, byte-identical to\n# round 1: the tracing walks the model\'s insides while it\n# hears the clip under the same question the outside tests\n# asked. Silence clips carry the tone question, the primary\n# task\'s question, exactly as round 1\'s negatives did, and\n# the manifest records the exact prompt on every clip.\nimport json, math\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nCLIPS = "/content/clips"\nPROMPTS = {\n    "tone": ("Is there the sound of a tone in this clip? "\n             "Answer yes or no."),\n    "speech": ("Is there the sound of speech in this clip? "\n               "Answer yes or no."),\n    "silence": ("Is there the sound of a tone in this clip? "\n                "Answer yes or no."),\n}\n\nclass FeedError(RuntimeError):\n    pass\n\ndef conversation_for(path, prompt):\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "audio", "audio": path},\n            {"type": "text", "text": prompt},\n        ],\n    }]\n\ndef feed_inputs(path, prompt, expect_silence=False):\n    # Run the feed path for one clip and build the model\'s\n    # inputs. Raises FeedError if the path returns no audio:\n    # a missing track stops the run, it is never padded. A\n    # track that decodes to all zeros also stops the run,\n    # unless the caller declares the clip silent by design\n    # (a generated true-silence clip): an all-zeros feed on\n    # a clip that should carry sound is a broken feed, not a\n    # quiet clip, and scoring it anyway is how silent\n    # failures get into the record.\n    if not os.path.exists(path):\n        raise FeedError(f"feed file missing: {path}")\n    conversation = conversation_for(path, prompt)\n    audios, images, videos = process_mm_info(\n        conversation, use_audio_in_video=False)\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if len(audio_np) == 0:\n        raise FeedError(\n            f"decoded track is empty for {path}")\n    if not expect_silence and not np.any(audio_np != 0):\n        raise FeedError(\n            f"decoded track is all zeros for {path}, on a clip "\n            f"that should carry sound (decoded length "\n            f"{len(audio_np)}, decoded rms "\n            f"{rms_db_np(audio_np):.2f} dB)")\n    text = processor.apply_chat_template(\n        conversation, add_generation_prompt=True, tokenize=False)\n    inputs = processor(text=text, audio=audios,\n                       return_tensors="pt", padding=True)\n    inputs = inputs.to(model.device).to(model.dtype)\n    return inputs, audio_np\n\ndef thinker_forward(inputs, want_attentions=False):\n    # One frozen pass through the thinker with every layer\'s\n    # numbers saved. The model is never trained, never asked\n    # to generate; this is a reading, not a conversation.\n    with torch.no_grad():\n        out = THINKER(**inputs, output_hidden_states=True,\n                      output_attentions=want_attentions)\n    return out\n\n# --- Run log: every print also lands in a file in the Drive\n# folder, so a stopped run\'s evidence never depends on the\n# notebook output area\'s scrolling.\nclass _Tee:\n    def __init__(self, *streams):\n        self.streams = streams\n    def write(self, s):\n        for st in self.streams:\n            try:\n                st.write(s)\n            except Exception:\n                pass\n        return len(s)\n    def flush(self):\n        for st in self.streams:\n            try:\n                st.flush()\n            except Exception:\n                pass\ntry:\n    _log_fh = open("/content/drive/MyDrive/tracing-qwen3b/"\n                   "property_run_log.txt", "w", buffering=1)\nexcept Exception:\n    _log_fh = open("/content/property_run_log.txt", "w",\n                   buffering=1)\nsys.stdout = _Tee(sys.stdout, _log_fh)\n\n# --- The token check -----------------------------------------------------------------\n# Round 2 fix 1. Round 1\'s gate picked the first candidate\n# id that decoded to the audio pad token and occurred at\n# least once, and it occurred exactly once: the probes may\n# have pooled over a single marker position instead of the\n# audio positions. This check settles the question from the\n# data, in the open, before any extraction:\n#\n#   * for one tone clip and one speech clip (the masters),\n#     print the FULL histogram of token ids in the processed\n#     input row: every id with count > 0, with the token\'s\n#     name and its decoded form, special tokens flagged;\n#   * print the processor\'s audio feature lengths (the\n#     input_features shape and the feature_attention_mask\n#     sums), the independent record of how many audio frames\n#     the processor built;\n#   * collect the audio-token candidates exactly as round 1\n#     did (tokenizer tokens whose text mentions audio, the\n#     config attributes, the converted pad id), print each\n#     candidate\'s count in the row, and DERIVE the pooling\n#     id from the histogram: the candidate with the highest\n#     count, the one that occurs at audio-frame scale;\n#   * both clips must derive the same id, and its count\n#     must be at least MIN_AUDIO_POSITIONS, or the step\n#     prints TOKEN CHECK FAILED and stops the run (exit 11)\n#     before any extraction;\n#   * one thinker forward on the tone clip then confirms\n#     the hidden states come back and fixes the layer count\n#     and hidden size the extraction will use.\n# Every line of the check is also collected and saved to\n# property_token_check.txt, pass or fail, so the histogram\n# is in the record either way.\nfrom collections import Counter\n\nMIN_AUDIO_POSITIONS = 10\nAUDIO_TOKEN_ID = None   # set by the token check, from the histogram\nN_STATES = None\nHIDDEN_SIZE = None\nTOKEN_CHECK_LINES = []\n\ndef tc(line=""):\n    print(line)\n    TOKEN_CHECK_LINES.append(line)\n\ndef save_token_check():\n    text = "\\n".join(TOKEN_CHECK_LINES) + "\\n"\n    def w(p):\n        with open(p, "w") as fh:\n            fh.write(text)\n    save_both("property_token_check.txt", w)\n\ndef token_check_fail(reason):\n    tc()\n    tc("!" * 70)\n    tc("TOKEN CHECK FAILED: " + reason)\n    tc("The pooling positions are not settled, so no extraction")\n    tc("has run and no later number in this run can be trusted.")\n    tc("The full histogram above is the evidence. Show this whole")\n    tc("output to Muse.")\n    tc("!" * 70)\n    save_token_check()\n    sys.exit(11)\n\ndef audio_candidates():\n    # The candidate audio ids, from the model\'s own\n    # tokenizer and config, never a guessed number. Same\n    # evidence round 1 gathered; what changed is how the\n    # winner is picked (see token_check).\n    tok = processor.tokenizer\n    vocab = tok.get_vocab()\n    audio_named = {t: i for t, i in vocab.items() if "audio" in t}\n    tc("tokenizer tokens whose text mentions audio:")\n    for t in sorted(audio_named):\n        tc(f"  {t!r} -> id {audio_named[t]}")\n    candidates = {}\n    pad_id = tok.convert_tokens_to_ids("<|audio_pad|>")\n    if pad_id is not None and int(pad_id) >= 0:\n        candidates[int(pad_id)] = "<|audio_pad|> (converted by name)"\n    for t, i in audio_named.items():\n        candidates[int(i)] = f"tokenizer token {t!r}"\n    for holder_name, holder in (\n            ("model.config", model.config),\n            ("thinker_config",\n             getattr(model.config, "thinker_config", None))):\n        if holder is None:\n            continue\n        for attr in ("audio_token_id", "audio_token_index"):\n            val = getattr(holder, attr, None)\n            if val is not None:\n                tc(f"  {holder_name}.{attr} = {val}")\n                candidates.setdefault(\n                    int(val), f"config {holder_name}.{attr}")\n    return candidates\n\ndef histogram_for(label, path, prompt, candidates):\n    # Feed one master clip and print its full token\n    # histogram plus the processor\'s audio feature lengths.\n    # Returns (inputs, chosen_id, chosen_count, positions).\n    tok = processor.tokenizer\n    try:\n        inputs, audio_np = feed_inputs(path, prompt)\n    except FeedError as e:\n        token_check_fail(\n            f"the {label} clip\'s feed failed inside the token "\n            f"check: {e}")\n    row = [int(x) for x in inputs.input_ids[0].tolist()]\n    tc("-" * 70)\n    tc(f"TOKEN HISTOGRAM: {label} clip")\n    tc(f"  file: {path}")\n    tc(f"  decoded waveform: {len(audio_np)} samples at 16 kHz "\n       f"({len(audio_np) / 16000:.2f} s)")\n    tc(f"  sequence length: {len(row)} positions")\n    counts = Counter(row)\n    tc(f"  distinct token ids: {len(counts)}")\n    tc("  full histogram, most common first (every id with "\n       "count > 0):")\n    special_ids = set(int(x) for x in tok.all_special_ids)\n    for tid, cnt in counts.most_common():\n        name = tok.convert_ids_to_tokens([tid])[0]\n        decoded = tok.decode([tid])\n        flag = " [special token]" if tid in special_ids else ""\n        tc(f"    id {tid}: count {cnt}, token name {name!r}, "\n           f"decodes to {decoded!r}{flag}")\n    feats = getattr(inputs, "input_features", None)\n    if feats is not None:\n        tc(f"  processor input_features shape: "\n           f"{tuple(feats.shape)} (batch, mel bins, audio frames)")\n    else:\n        tc("  processor input_features: not present on the inputs")\n    fam = getattr(inputs, "feature_attention_mask", None)\n    if fam is not None:\n        tc(f"  processor feature_attention_mask shape: "\n           f"{tuple(fam.shape)}, true frames per row: "\n           f"{[int(x) for x in fam.sum(dim=-1).tolist()]}")\n    else:\n        tc("  processor feature_attention_mask: not present")\n    tc("  candidate audio ids, counted in this row:")\n    cand_counts = {}\n    for cid, why in sorted(candidates.items()):\n        cnt = row.count(cid)\n        cand_counts[cid] = cnt\n        tc(f"    candidate id {cid} ({why}): count {cnt}, "\n           f"decodes to {tok.decode([cid])!r}")\n    chosen = max(cand_counts, key=lambda c: cand_counts[c])\n    positions = [i for i, x in enumerate(row) if x == chosen]\n    tc(f"  DERIVED audio token id for the {label} clip: {chosen} "\n       f"({candidates[chosen]}), decodes to "\n       f"{tok.decode([chosen])!r}")\n    tc(f"  audio positions: {len(positions)} of {len(row)}; "\n       f"first {positions[:5]}, last {positions[-5:]}")\n    return inputs, chosen, cand_counts[chosen], positions\n\ndef token_check():\n    global AUDIO_TOKEN_ID, N_STATES, HIDDEN_SIZE\n    tc("=" * 70)\n    tc("TOKEN CHECK: settle the audio positions from the "\n       "histogram, before any extraction")\n    tc(f"plausibility floor: the derived audio-position count "\n       f"must be at least {MIN_AUDIO_POSITIONS}")\n    candidates = audio_candidates()\n    tone_path = os.path.join(CLIPS, "codec_tone_ref_raw.wav")\n    speech_path = os.path.join(CLIPS, "codec_speech_raw.wav")\n    tone_inputs, tone_id, tone_count, _ = histogram_for(\n        "tone", tone_path, PROMPTS["tone"], candidates)\n    _, speech_id, speech_count, _ = histogram_for(\n        "speech", speech_path, PROMPTS["speech"], candidates)\n    if tone_id != speech_id:\n        token_check_fail(\n            f"the two clips derive different audio token ids "\n            f"(tone {tone_id}, speech {speech_id}); the pooling "\n            f"positions cannot be settled on a split answer")\n    if tone_count < MIN_AUDIO_POSITIONS \\\n            or speech_count < MIN_AUDIO_POSITIONS:\n        token_check_fail(\n            f"the derived audio-position count is implausible "\n            f"(tone {tone_count}, speech {speech_count}, floor "\n            f"{MIN_AUDIO_POSITIONS}); the histogram does not "\n            f"show an id occurring at audio-frame scale")\n    AUDIO_TOKEN_ID = tone_id\n    tc("-" * 70)\n    tc(f"AUDIO TOKEN ID SETTLED: {AUDIO_TOKEN_ID}, from the "\n       f"histogram (tone count {tone_count}, speech count "\n       f"{speech_count})")\n    out = thinker_forward(tone_inputs)\n    hs = getattr(out, "hidden_states", None)\n    if hs is None or len(hs) == 0:\n        token_check_fail(\n            "the thinker returned no hidden states on the tone "\n            "clip; there would be nothing to pool")\n    N_STATES = len(hs)\n    HIDDEN_SIZE = int(hs[0].shape[2])\n    tc(f"hidden states returned: {N_STATES} tensors (layer 0 is "\n       f"the embedding output, then one per thinker layer)")\n    tc(f"thinker layers: {N_STATES - 1}")\n    tc(f"hidden size: {HIDDEN_SIZE}")\n    tc(f"tone sequence length: {int(hs[0].shape[1])}")\n    del out, tone_inputs\n    free_memory()\n    tc("TOKEN CHECK PASSED: the audio positions are derived "\n       "from the histogram, counted, and the same for both "\n       "clips. Every later number in this run pools over "\n       "these positions.")\n    save_token_check()\n\ntoken_check()\n\ndef audio_positions(input_ids_row):\n    row = [int(x) for x in input_ids_row]\n    return [i for i, x in enumerate(row) if x == AUDIO_TOKEN_ID]\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, and the token check")\n    print("all passed. No clips were generated and no extraction ran.")\n    print("Run Step 4 for the battery.")\n    sys.exit(0)\n\n# --- Probe clip generation -------------------------------------------------------------\n# Round 2 fix 2. Fixed seed, so the run can be repeated\n# exactly and the split can be re-derived. 420 clips,\n# 360 per class (tone, speech, silence), each exactly\n# 4.0 s at 16 kHz mono (64000 samples), fed later as raw\n# WAV: no MP4, no codec anywhere.\n#\n# Variation inside each class, so a helper must read the\n# sound and not memorize one waveform:\n#   tone: a 0.5 s sine burst, frequency jittered 950 to\n#     1050 Hz, start 0.5 to 3.0 s, level -24 to -14 dBFS.\n#   speech: the probe sentence from the master, tape-style\n#     speed factor 0.92 to 1.08 (the resample shifts pitch\n#     with speed, the small speed/pitch variation), gain\n#     -6 to +3 dB, start offset up to 1.5 s.\n#   silence: nothing in the foreground at all.\n# Every clip, all classes alike, sits on a background drawn\n# from one shared scheme: plain digital silence, with a\n# one-in-six chance of a near-silence white-noise floor at\n# -75 to -65 dBFS. The silence class is therefore true\n# digital silence plus a few near-silence noise-floor\n# variants, and the background can never be a class cue,\n# because every class draws from the same scheme.\n#\n# The split is fixed by seed: 240 train / 120 held-out test\n# per class, stratified, the same split for every layer and\n# both tasks. split_ranks below is the single source of\n# truth: generation stamps each clip\'s split in the\n# manifest, and the probe step re-derives the same ranks and\n# asserts the two agree.\n#\n# Every clip is measured before anything uses it (sample\n# count, whole-clip RMS, per-second RMS, waveform SHA-256)\n# and printed: the feed proof. A failed measurement stops\n# the run (exit 9).\nimport csv, hashlib\n\nSEED = 20261105\nN_TRAIN = 240\nSR = 16000\nCLIP_SAMPLES = SR * 4\nCLASS_NUMBER = {"tone": 1, "speech": 2, "silence": 3}\nCLIP_DIRS = ["/content/drive/MyDrive/tracing-qwen3b/property_clips",\n             "/content/property_clips"]\nfor d in CLIP_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\nMANIFEST_FIELDS = [\'clip_name\', \'group\', \'split\', \'file\', \'freq_hz\', \'start_s\', \'burst_s\', \'level_dbfs\', \'gain_db\', \'speed_factor\', \'floor_dbfs\', \'rms_dbfs\', \'wave_sha256\', \'prompt\', \'pitch_band\', \'onset_band\']\n\ndef split_ranks(class_name):\n    # The within-class ranks (0-based, in generation order)\n    # that belong to train and to test for one class.\n    rng = np.random.default_rng(SEED + 1009 * CLASS_NUMBER[class_name])\n    order = rng.permutation(360)\n    return (set(int(x) for x in order[:N_TRAIN]),\n            set(int(x) for x in order[N_TRAIN:]))\n\ndef rms_db_np(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x.astype(np.float64)))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef db_to_amp(db):\n    return float(10.0 ** (db / 20.0))\n\ndef wave_hash(wave):\n    return hashlib.sha256(\n        np.asarray(wave, dtype=np.float32).tobytes()).hexdigest()\n\ndef background(rng):\n    # One draw from the shared background scheme: (wave,\n    # floor level in dBFS or None for digital silence).\n    wave = np.zeros(CLIP_SAMPLES, dtype=np.float32)\n    if rng.random() < 1.0 / 6.0:\n        level = float(rng.uniform(-75.0, -65.0))\n        noise = rng.standard_normal(CLIP_SAMPLES).astype(np.float32)\n        cur = float(np.sqrt(np.mean(np.square(\n            noise.astype(np.float64)))))\n        wave = (noise * (db_to_amp(level) / cur)).astype(np.float32)\n        return wave, round(level, 2)\n    return wave, None\n\ndef write_clip(name, wave):\n    wave = np.asarray(wave, dtype=np.float32)\n    assert len(wave) == CLIP_SAMPLES, \\\n        f"clip {name} has {len(wave)} samples, not {CLIP_SAMPLES}"\n    local = os.path.join("/content/property_clips", name)\n    drive = os.path.join(\n        "/content/drive/MyDrive/tracing-qwen3b/property_clips", name)\n    # ALWAYS write the wave in hand, to both homes, overwriting\n    # whatever is there. Never copy an existing file into place:\n    # /content is fresh each session but the Drive folder\n    # persists across runs, and an earlier run\'s file under the\n    # same name can hold a different wave (the draw sequence\n    # moved when the builder changed). That stale-file reuse is\n    # exactly what stopped the Nov 3 diagnostic run at clip\n    # 413: prop_silence_53\'s row described a -67.5 dB noise\n    # floor while the file on disk was an older run\'s digital\n    # silence. The read-back check in finish_row now proves\n    # the file on disk is the wave in hand before extraction\n    # ever sees it.\n    sf.write(local, wave, SR, subtype="PCM_16")\n    try:\n        sf.write(drive, wave, SR, subtype="PCM_16")\n    except Exception:\n        pass\n    return local\n\ndef speech_segment():\n    # The sentence inside the speech master: the region whose\n    # short-window RMS rises above -45 dBFS, with a small pad.\n    # Same cut as round 1.\n    path = os.path.join(CLIPS, "codec_speech_raw.wav")\n    wave, _ = librosa.load(path, sr=SR, mono=True)\n    wave = np.asarray(wave, dtype=np.float32)\n    win = SR // 100  # 10 ms\n    n = len(wave) // win\n    rms = np.array([rms_db_np(wave[i * win:(i + 1) * win])\n                    for i in range(n)])\n    above = np.where(rms > -45.0)[0]\n    if len(above) == 0:\n        raise RuntimeError(\n            "no active region found in the speech master; the "\n            "speech positives cannot be built")\n    start = max(0, int(above[0] * win) - SR // 20)\n    end = min(len(wave), int((above[-1] + 1) * win) + SR // 20)\n    seg = wave[start:end]\n    print(f"speech segment: {len(seg) / SR:.2f} s cut from the "\n          f"master at {start / SR:.2f} s to {end / SR:.2f} s")\n    if len(seg) > CLIP_SAMPLES:\n        seg = seg[:CLIP_SAMPLES]\n        print("speech segment trimmed to the 4.0 s window")\n    return seg\n\ndef tape_resample(x, speed):\n    # Tape-style speed change: playing the segment faster or\n    # slower, which shifts its pitch by the same factor. The\n    # round 2 speech variation (speed 0.92 to 1.08, about a\n    # semitone and a half either way at the extremes).\n    n_out = max(1, int(round(len(x) / speed)))\n    idx = np.linspace(0, len(x) - 1, n_out)\n    return np.interp(idx, np.arange(len(x)), x).astype(np.float32)\n\ndef finish_row(rows, name, group, rank, wave, prompt, extra):\n    # Peak guard, split stamp, measurement, and the printed\n    # feed proof for one generated clip.\n    peak = float(np.max(np.abs(wave))) if len(wave) else 0.0\n    if peak > 0.99:\n        wave = (wave * (0.99 / peak)).astype(np.float32)\n        print(f"  {name}: peak {peak:.2f} over the guard; scaled "\n              f"to a 0.99 peak")\n    path = write_clip(name, wave)\n    # Read-back proof: the file on disk must be the wave in\n    # hand (PCM_16 quantisation moves the RMS by far less\n    # than 1 dB at these levels; digital silence reads back\n    # at exactly -120). A mismatch stops generation here,\n    # where the cause is visible, instead of surfacing as a\n    # feed hard stop deep inside extraction.\n    _back, _ = sf.read(path, dtype="float32")\n    if len(_back) != len(wave) or abs(rms_db_np(_back)\n                                      - rms_db_np(wave)) > 1.0:\n        print()\n        print("!" * 70)\n        print(f"CLIP WRITE CHECK FAILED: {name} on disk does "\n              f"not match the generated wave (disk rms "\n              f"{rms_db_np(_back):.2f} dB vs wave rms "\n              f"{rms_db_np(wave):.2f} dB). Nothing was "\n              f"extracted. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(9)\n    train_ranks, _ = split_ranks(group)\n    split = "train" if rank in train_ranks else "test"\n    row = {"clip_name": name[:-4], "group": group, "split": split,\n           "file": path, "wave_sha256": wave_hash(wave),\n           "rms_dbfs": round(rms_db_np(wave), 2), "prompt": prompt,\n           "_expect_silence": bool(group == "silence"\n                                   and extra.get("floor_dbfs") in ("", None))}\n    row.update(extra)\n    split = row["split"]\n    sec = [rms_db_np(wave[k * SR:(k + 1) * SR]) for k in range(4)]\n    print(f"  {row[\'clip_name\']:<14} group={group:<7} split={split:<5} "\n          f"samples={len(wave)} rms={row[\'rms_dbfs\']:.2f} dB "\n          f"per-second [{\', \'.join(f\'{v:.1f}\' for v in sec)}] dB "\n          f"sha256={row[\'wave_sha256\'][:16]}")\n    if len(wave) != CLIP_SAMPLES:\n        print()\n        print("!" * 70)\n        print(f"CLIP MEASUREMENT FAILED: {row[\'clip_name\']} has "\n              f"{len(wave)} samples, not {CLIP_SAMPLES}. The "\n              f"generated set is not what the plan says. Nothing "\n              f"was extracted. Show this whole output to Muse.")\n        print("!" * 70)\n        sys.exit(9)\n    rows.append(row)\n\ndef generate_clips():\n    # The Handover Correction step 5 stimulus set. Factorial\n    # tones: pitch (500 vs 1500 Hz) crossed with onset (0.5 vs\n    # 2.5 s), 90 instances per cell, 360 tones; 60 silence\n    # clips (the shared background scheme alone). Every\n    # instance is a fresh draw (pitch jitter x0.98 to 1.02,\n    # onset jitter +-0.1 s, level -20 +-1 dBFS, random phase),\n    # so every waveform is a unique identity; the split is by\n    # instance within each cell (60 train / 30 test), stamped\n    # here and asserted unique by hash below.\n    rng = np.random.default_rng(SEED)\n    rows = []\n    cells = [("low", 500.0, "early", 0.5), ("low", 500.0, "late", 2.5),\n             ("high", 1500.0, "early", 0.5), ("high", 1500.0, "late", 2.5)]\n    for ci, (pband, pbase, oband, obase) in enumerate(cells):\n        order = rng.permutation(90)\n        for i in range(90):\n            freq = float(pbase * rng.uniform(0.98, 1.02))\n            start_s = float(obase + rng.uniform(-0.1, 0.1))\n            level = float(-20.0 + rng.uniform(-1.0, 1.0))\n            phase = float(rng.uniform(0.0, 2 * np.pi))\n            burst_s = 0.5\n            wave, floor_db = background(rng)\n            n_burst = int(burst_s * SR)\n            t = np.arange(n_burst) / SR\n            burst = (db_to_amp(level)\n                     * np.sin(2 * np.pi * freq * t + phase)).astype(np.float32)\n            edge = SR // 200\n            ramp = 0.5 - 0.5 * np.cos(\n                np.linspace(0, np.pi, edge)).astype(np.float32)\n            burst[:edge] *= ramp\n            burst[-edge:] *= ramp[::-1]\n            s0 = int(start_s * SR)\n            wave[s0:s0 + n_burst] += burst\n            split = "train" if int(order[i]) < 60 else "test"\n            finish_row(rows, f"prop_tone_{ci}_{i:02d}.wav", "tone", i, wave,\n                       PROMPTS["tone"],\n                       {"freq_hz": round(freq, 1),\n                        "start_s": round(start_s, 3), "burst_s": burst_s,\n                        "level_dbfs": round(level, 2), "gain_db": "",\n                        "speed_factor": "",\n                        "floor_dbfs": floor_db if floor_db is not None else "",\n                        "pitch_band": pband, "onset_band": oband,\n                        "split": split})\n    # Silence: the shared background scheme returns pure\n    # digital silence for most draws, so identical silence\n    # waveforms are expected and legitimate (silence is one\n    # stimulus). The discipline that matters is split\n    # hygiene: identical waveforms are grouped and a group\n    # sits entirely on one side of the split, so no stimulus\n    # is ever on both sides. Draw all 60 first, group by\n    # hash, then stamp.\n    sil_draws = []\n    for i in range(60):\n        wave, floor_db = background(rng)\n        sil_draws.append((i, wave, floor_db))\n    groups = {}\n    for i, wave, floor_db in sil_draws:\n        groups.setdefault(wave_hash(wave), []).append((i, wave, floor_db))\n    gorder = list(groups.keys())\n    rng.shuffle(gorder)\n    train_hashes = set()\n    n_train = 0\n    for h in gorder:\n        if n_train >= 40:\n            break\n        train_hashes.add(h)\n        n_train += len(groups[h])\n    for h in gorder:\n        split = "train" if h in train_hashes else "test"\n        for i, wave, floor_db in groups[h]:\n            finish_row(rows, f"prop_silence_{i:02d}.wav", "silence", i, wave,\n                       PROMPTS["silence"],\n                       {"freq_hz": "", "start_s": "", "burst_s": "",\n                        "level_dbfs": "", "gain_db": "", "speed_factor": "",\n                        "floor_dbfs": floor_db if floor_db is not None else "",\n                        "pitch_band": "", "onset_band": "",\n                        "split": split})\n    print("-" * 70)\n    tone_hashes = [r["wave_sha256"] for r in rows if r["group"] == "tone"]\n    assert len(set(tone_hashes)) == len(tone_hashes), \\\n        "duplicate tone waveform hashes; the property identities must be unique"\n    sil_by_split = {}\n    for r in rows:\n        if r["group"] == "silence":\n            sil_by_split.setdefault(r["wave_sha256"], set()).add(r["split"])\n    assert all(len(v) == 1 for v in sil_by_split.values()), \\\n        "a silence waveform sits on both sides of the split"\n    assert not (set(tone_hashes) & set(sil_by_split)), \\\n        "a tone waveform hash collides with a silence waveform"\n    print(f"generated {len(rows)} clips; {len(tone_hashes)} tone waveforms, "\n          f"all unique; silence in {len(sil_by_split)} unique waveforms, "\n          f"each entirely on one side of the split")\n    from collections import Counter\n    splits = Counter((r["group"], r["split"]) for r in rows)\n    print(f"  tone: {splits[(\'tone\', \'train\')]} train, {splits[(\'tone\', \'test\')]} test")\n    print(f"  silence: {splits[(\'silence\', \'train\')]} train, {splits[(\'silence\', \'test\')]} test")\n    def write_manifest(p):\n        with open(p, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=MANIFEST_FIELDS)\n            w.writeheader()\n            for r in rows:\n                w.writerow({k: r.get(k, "") for k in MANIFEST_FIELDS})\n    save_both("property_manifest.csv", write_manifest)\n    return rows\n\n# --- Extraction --------------------------------------------------------------------------\n# One frozen forward pass per clip. Per layer, the vectors\n# at the VERIFIED audio positions (the token check\'s id)\n# are averaged into one vector per clip and stored. The\n# attention subset (the first 6 tone and first 6 silence\n# clips, in manifest order, the round 1 subset shape) gets\n# a second forward with attentions returned; for EVERY\n# layer we record the mean attention from the final sequence\n# position to the verified audio positions. Features are\n# saved as float16 (property_features.npz) with the file\n# size printed.\nFEATURES_NAME = "property_features.npz"\n\ndef find_features():\n    for p in out_paths(FEATURES_NAME):\n        if os.path.exists(p):\n            return p\n    return None\n\ndef extract(rows):\n    n_clips = len(rows)\n    feats = np.zeros((N_STATES, n_clips, HIDDEN_SIZE),\n                     dtype=np.float32)\n    feats_final = np.zeros((N_STATES, n_clips, HIDDEN_SIZE),\n                           dtype=np.float32)\n    audio_counts = np.zeros(n_clips, dtype=np.int64)\n    seq_lens = np.zeros(n_clips, dtype=np.int64)\n    attention_rows = []\n    subset = set()\n    tone_seen = sil_seen = 0\n    for idx, r in enumerate(rows):\n        if r["group"] == "tone" and tone_seen < 6:\n            subset.add(idx)\n            tone_seen += 1\n        elif r["group"] == "silence" and sil_seen < 6:\n            subset.add(idx)\n            sil_seen += 1\n    for idx, r in enumerate(rows):\n        expect_silence = bool(r.get("_expect_silence", False))\n        try:\n            inputs, audio_np = feed_inputs(\n                r["file"], r["prompt"],\n                expect_silence=expect_silence)\n        except FeedError as e:\n            print()\n            print("!" * 70)\n            print("EXTRACTION FEED HARD STOP:")\n            print(str(e))\n            try:\n                _w, _sr = sf.read(r["file"], dtype="float32")\n                print(f"  on-disk check for {r[\'file\']}: "\n                      f"{len(_w)} samples at {_sr} Hz, rms "\n                      f"{rms_db_np(np.asarray(_w)):.2f} dB, "\n                      f"any nonzero: {bool(np.any(_w != 0))}, "\n                      f"expect_silence flag: {expect_silence}")\n            except Exception as _e2:\n                print(f"  on-disk check failed: {str(_e2)[:160]}")\n            print(f"Clip {idx + 1} of {n_clips} ({r[\'clip_name\']}) "\n                  f"was not extracted, and the run stops here. No")\n            print("padding was substituted. Show this whole output")\n            print("to Muse.")\n            print("!" * 70)\n            sys.exit(7)\n        row_ids = inputs.input_ids[0].tolist()\n        pos = audio_positions(row_ids)\n        if len(pos) == 0:\n            print()\n            print("!" * 70)\n            print(f"EXTRACTION HARD STOP: clip {r[\'clip_name\']} has "\n                  f"zero audio-token positions under the token "\n                  f"check\'s derived id {AUDIO_TOKEN_ID}. The check "\n                  f"found them on both masters, so this clip\'s feed "\n                  f"differs somehow. No vector was faked. Show this "\n                  f"output to Muse.")\n            print("!" * 70)\n            sys.exit(11)\n        out = thinker_forward(inputs)\n        hs = out.hidden_states\n        pos_t = torch.tensor(pos, dtype=torch.long)\n        for li in range(N_STATES):\n            layer_vec = hs[li][0].index_select(0, pos_t.to(\n                hs[li].device)).mean(dim=0)\n            feats[li, idx] = layer_vec.float().cpu().numpy()\n            feats_final[li, idx] = hs[li][0][-1].float().cpu().numpy()\n        audio_counts[idx] = len(pos)\n        seq_lens[idx] = int(hs[0].shape[1])\n        print(f"extracted {idx + 1}/{n_clips}: {r[\'clip_name\']} "\n              f"(group={r[\'group\']}, audio tokens={len(pos)}, "\n              f"seq len={seq_lens[idx]})")\n        del out, inputs\n        if idx in subset:\n            inputs2, _ = feed_inputs(\n                r["file"], r["prompt"],\n                expect_silence=expect_silence)\n            out2 = thinker_forward(inputs2, want_attentions=True)\n            atts = getattr(out2, "attentions", None)\n            if atts is None or len(atts) == 0:\n                print(f"WARNING: attention subset clip "\n                      f"{r[\'clip_name\']} returned no attentions; "\n                      f"its rows will be missing from the attention")\n                print("CSV, and this warning is the record of why.")\n            else:\n                for li in range(len(atts)):\n                    a = atts[li][0].float()  # heads, seq, seq\n                    val = float(a[:, -1, :][:, pos].mean().item())\n                    attention_rows.append({\n                        "clip": r["clip_name"], "layer": li,\n                        "attention_to_audio": round(val, 6)})\n                print(f"attention read: {r[\'clip_name\']} "\n                      f"(all {len(atts)} layers)")\n            del out2, inputs2\n        free_memory()\n    extracted = int(np.count_nonzero(audio_counts))\n    print(f"extraction count: {extracted} of {n_clips} clips")\n    if extracted != n_clips:\n        print()\n        print("!" * 70)\n        print(f"EXTRACTION COUNT MISMATCH: {extracted} clips have "\n              f"features, the manifest lists {n_clips}. The run")\n        print("stops; partial features are not saved as a result.")\n        print("!" * 70)\n        sys.exit(10)\n    groups = np.array([r["group"] for r in rows])\n    names = np.array([r["clip_name"] for r in rows])\n\n    def write_npz(p):\n        np.savez(p, features=feats.astype(np.float16),\n                 features_final=feats_final.astype(np.float16),\n                 groups=groups, clip_names=names,\n                 audio_token_counts=audio_counts, seq_lens=seq_lens)\n    saved = save_both(FEATURES_NAME, write_npz)\n    for p in saved:\n        print(f"features file (float16): {p}, "\n              f"{os.path.getsize(p) / 1e6:.1f} MB")\n    return feats, feats_final, groups, names, attention_rows\n\n# --- Battery flow ---------------------------------------------------------------------------\nrows = generate_clips()\nattention_rows = []\nfeats = groups = names = None\nexisting = find_features()\nif existing is not None:\n    try:\n        z = np.load(existing, allow_pickle=False)\n        f0 = z["features"]\n        g0 = [str(x) for x in z["groups"]]\n        n0 = [str(x) for x in z["clip_names"]]\n        if f0.shape == (N_STATES, len(rows), HIDDEN_SIZE) \\\n                and n0 == [r["clip_name"] for r in rows] \\\n                and g0 == [r["group"] for r in rows]:\n            feats = f0.astype(np.float32)\n            groups, names = z["groups"], z["clip_names"]\n            print(f"resume: {existing} already holds all "\n                  f"{len(rows)} clips with matching names and "\n                  f"groups; extraction skipped, going straight "\n                  f"to the probes. The attention readout is not "\n                  f"re-run on a resume; its CSV from the first "\n                  f"pass stands.")\n        else:\n            print(f"resume: {existing} exists but does not match "\n                  f"this manifest (shape {f0.shape}); extracting "\n                  f"fresh and overwriting it.")\n    except Exception as e:\n        print(f"resume: could not read {existing} ({str(e)[:120]}); "\n              f"extracting fresh.")\nif feats is None:\n    feats, feats_final, groups, names, attention_rows = extract(rows)\n\nif attention_rows:\n    def write_attention(p):\n        with open(p, "w", newline="") as fh:\n            w = csv.DictWriter(fh, fieldnames=[\'clip\', \'layer\', \'attention_to_audio\'])\n            w.writeheader()\n            w.writerows(attention_rows)\n    save_both("property_attention.csv", write_attention)\n\n# --- The helpers (probes), round 2 readout -----------------------------------------------------\n# Round 2 fix 3. Per layer, one vector per clip: the mean\n# over the verified audio positions. Two readouts per layer\n# per task, both small on purpose:\n#   plain: standardized features, logistic regression with\n#     strong regularization (C=0.01);\n#   pca64: standardized features reduced to 64 principal\n#     components, then the same probe.\n# Tasks unchanged from round 1: tone vs silence, speech vs\n# silence. The split is the seed-fixed split from\n# generation: 240 train / 120 test per class, the same at\n# every layer, re-derived here from split_ranks and asserted\n# against the manifest stamps. Every helper is shadowed by\n# a shuffled-label control: 5 shuffles of the training\n# labels, each trained and tested identically to the real\n# helper, reported as their mean (and standard deviation).\n# A control that beats chance voids the run under the plan.\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.pipeline import make_pipeline\n\nPROBE_FIELDS = [\'state\', \'task\', \'readout\', \'n_train\', \'n_test\', \'accuracy\', \'shuffled_accuracy_mean\', \'shuffled_accuracy_std\', \'shuffled_p95\']\nN_SHUFFLES = 5\nPROBE_C = 0.01\nPCA_COMPONENTS = 64\ngroup_list = [str(g) for g in groups]\n\n# Splits and labels come from the manifest stamps written at\n# generation: per-cell instance splits for the tones, so no\n# waveform identity appears on both sides of any task.\ntone_tr = [i for i, r in enumerate(rows)\n           if r["group"] == "tone" and r["split"] == "train"]\ntone_te = [i for i, r in enumerate(rows)\n           if r["group"] == "tone" and r["split"] == "test"]\nsil_tr = [i for i, r in enumerate(rows)\n          if r["group"] == "silence" and r["split"] == "train"]\nsil_te = [i for i, r in enumerate(rows)\n          if r["group"] == "silence" and r["split"] == "test"]\nprint(f"tone split: {len(tone_tr)} train, {len(tone_te)} test; "\n      f"silence split: {len(sil_tr)} train, {len(sil_te)} test")\n\ndef band_labels(idxs, key, positive):\n    return np.array([1 if rows[i][key] == positive else 0\n                     for i in idxs])\n\nTASKS = {\n    "pitch_high_vs_low": (\n        tone_tr, tone_te,\n        band_labels(tone_tr, "pitch_band", "high"),\n        band_labels(tone_te, "pitch_band", "high")),\n    "onset_late_vs_early": (\n        tone_tr, tone_te,\n        band_labels(tone_tr, "onset_band", "late"),\n        band_labels(tone_te, "onset_band", "late")),\n    "presence_tone_vs_silence": (\n        tone_tr + sil_tr, tone_te + sil_te,\n        np.array([1] * len(tone_tr) + [0] * len(sil_tr)),\n        np.array([1] * len(tone_te) + [0] * len(sil_te))),\n}\nREADOUTS = {"audio_plain": (feats, "plain"),\n            "audio_pca64": (feats, "pca64"),\n            "final_plain": (feats_final, "plain"),\n            "final_pca64": (feats_final, "pca64")}\n\ndef fit_readout(X_train, y_train, readout):\n    if readout == "plain":\n        pipe = make_pipeline(\n            StandardScaler(),\n            LogisticRegression(C=PROBE_C, max_iter=1000))\n    elif readout == "pca64":\n        pipe = make_pipeline(\n            StandardScaler(),\n            PCA(n_components=PCA_COMPONENTS, random_state=SEED),\n            LogisticRegression(C=PROBE_C, max_iter=1000))\n    else:\n        raise ValueError(f"unknown readout {readout}")\n    pipe.fit(X_train, y_train)\n    return pipe\n\nprobe_rows = []\nfor task, (tr_idx, te_idx, y_train, y_test) in TASKS.items():\n    for rname, (F, rtype) in READOUTS.items():\n        if task == "presence_tone_vs_silence" and rname.startswith("final"):\n            continue\n        for li in range(N_STATES):\n            X = F[li]\n            clf = fit_readout(X[tr_idx], y_train, rtype)\n            acc = float(clf.score(X[te_idx], y_test))\n            probe_rows.append({\n                "state": li, "task": task, "readout": rname,\n                "n_train": len(tr_idx), "n_test": len(te_idx),\n                "accuracy": round(acc, 4),\n                "shuffled_accuracy_mean": "",\n                "shuffled_accuracy_std": "",\n                "shuffled_p95": ""})\n            print(f"probe {task} {rname} state {li}: "\n                  f"accuracy {acc:.3f}")\n\ndef write_probes(p):\n    with open(p, "w", newline="") as fh:\n        w = csv.DictWriter(fh, fieldnames=PROBE_FIELDS)\n        w.writeheader()\n        w.writerows(probe_rows)\nsave_both("property_probe_results.csv", write_probes)\nprint("headline probe table saved before the controls run")\n\n# Presence controls: 5 shuffles, as round 2, audio readouts.\ntr_idx, te_idx, y_train, y_test = TASKS["presence_tone_vs_silence"]\nfor rname in ("audio_plain", "audio_pca64"):\n    F, rtype = READOUTS[rname]\n    for li in range(N_STATES):\n        X = F[li]\n        shuf = []\n        for s in range(N_SHUFFLES):\n            rng_s = np.random.default_rng(SEED + 500000 + li * 1000 + s)\n            clf_s = fit_readout(X[tr_idx], rng_s.permutation(y_train), rtype)\n            shuf.append(float(clf_s.score(X[te_idx], y_test)))\n        for r in probe_rows:\n            if (r["task"] == "presence_tone_vs_silence"\n                    and r["readout"] == rname and r["state"] == li):\n                r["shuffled_accuracy_mean"] = round(float(np.mean(shuf)), 4)\n                r["shuffled_accuracy_std"] = round(float(np.std(shuf)), 4)\n                r["shuffled_p95"] = round(float(np.max(shuf)), 4)\n    print(f"presence controls done for {rname} (5 shuffles per state)")\n\n# Property nulls: 200 permutations in the focus band (states\n# 16 to 25, the outputs of decoder layers 15 to 24), pca64\n# readouts at both positions.\nFOCUS_STATES = list(range(16, 26))\nN_PERMS = 200\nfor task in ("pitch_high_vs_low", "onset_late_vs_early"):\n    tr_idx, te_idx, y_train, y_test = TASKS[task]\n    for rname in ("audio_pca64", "final_pca64"):\n        F, rtype = READOUTS[rname]\n        for li in FOCUS_STATES:\n            X = F[li]\n            nulls = []\n            for s in range(N_PERMS):\n                rng_p = np.random.default_rng(\n                    SEED + 900000 + li * 10000 + s * 17\n                    + (0 if rname == "audio_pca64" else 5000)\n                    + (0 if task.startswith("pitch") else 2500))\n                clf_s = fit_readout(X[tr_idx],\n                                    rng_p.permutation(y_train), rtype)\n                nulls.append(float(clf_s.score(X[te_idx], y_test)))\n            for r in probe_rows:\n                if (r["task"] == task and r["readout"] == rname\n                        and r["state"] == li):\n                    r["shuffled_accuracy_mean"] = round(float(np.mean(nulls)), 4)\n                    r["shuffled_accuracy_std"] = round(float(np.std(nulls)), 4)\n                    r["shuffled_p95"] = round(float(np.percentile(nulls, 95)), 4)\n            print(f"nulls {task} {rname} state {li}: mean "\n                  f"{np.mean(nulls):.3f}, p95 {np.percentile(nulls, 95):.3f} "\n                  f"({N_PERMS} permutations)")\nsave_both("property_probe_results.csv", write_probes)\n\n# --- The plain-language tail ---------------------------------------------------------------------\nprint()\nprint("=" * 70)\nprint("READOUT (mechanical, no verdict beyond the numbers)")\nfor task in ("pitch_high_vs_low", "onset_late_vs_early"):\n    for rname in ("audio_pca64", "final_pca64"):\n        band = [r for r in probe_rows if r["task"] == task\n                and r["readout"] == rname and r["state"] in FOCUS_STATES]\n        accs = [r["accuracy"] for r in band]\n        p95s = [r["shuffled_p95"] for r in band]\n        print(f"  {task} [{rname}], states 16 to 25: accuracy "\n              f"min {min(accs):.3f}, max {max(accs):.3f}; "\n              f"null p95 max {max(p95s):.3f}")\nprint("  state k is the output of decoder layer k-1 (state 0 is")\nprint("  the embedding output); the focus band, states 16 to 25,")\nprint("  is decoder layers 15 to 24. A probe reading a property")\nprint("  shows the property is present in readable form at that")\nprint("  state; it does not show the model uses it.")\nprint("=" * 70)\nprint("battery done: token check passed, clips generated and")\nprint("measured, features extracted and counted, property")\nprint("helpers trained, focus-band permutation nulls run.")\nprint("Outputs: property_token_check.txt, property_manifest.csv,")\nprint("property_features.npz (float16), property_probe_results.csv,")\nprint("property_attention.csv.")\n'
with open("/content/qwen_property_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_property_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, token check --------------------------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, or the token check failed, and the run
# stops in this cell. The first check run downloads the model
# files once; that takes a while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_property_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, or the token check.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, token check")
print("passed with the audio positions derived and counted. Go on to Step 4.")

## Step 4 · Run the tracing battery

The next cell runs the runner in **battery mode**: proof,
candidates, and the token check again, then four stages.

**Clip generation.** 420 clips are synthesized with the
fixed seed 20261104, each exactly 4.0 seconds at 16 kHz:
360 tone (a 0.5 s burst, frequency jittered between 950
and 1050 Hz, start between 0.5 and 3.0 s, level between
-24 and -14 dBFS), 360 speech (the one probe sentence,
"The blue kettle boils at seven forty two.", rendered
from the master, tape-style speed between 0.92 and 1.08, gain
between -6 and +3 dB, start offset up to 1.5 s), and 360
silence (no foreground at all). Every clip sits on the
shared background scheme: digital silence, with a
one-in-six chance of a near-silence noise floor between
-75 and -65 dBFS. Every clip is measured and printed
(sample count, whole-clip RMS, per-second RMS, waveform
SHA-256) before anything uses it; a clip that fails its
measurement stops the run. The split is stamped per clip
in `property_manifest.csv`: 240 train and 120 test per
class, fixed by the seed.

**Extraction.** Each clip goes through the frozen model
once, sound alone, under the Series Two presence question
for its class. Any feed that returns no audio, or decodes
to zeros on a clip that should carry sound, stops the run
on the spot (exit code 7): nothing is padded and nothing
is scored on a broken feed. Per layer, the vectors at the
verified audio positions are averaged into one vector per
clip and saved to `property_features.npz` as float16, with
the file size printed. The extraction count must equal
the manifest count, 420, or the run stops (exit code
10). If the features file already holds all 420 clips
with matching names, extraction is skipped and the run
goes straight to the helpers.

**The helpers.** Per layer and per task (tone vs silence,
speech vs silence), two readouts train on 240 positives
plus 240 negatives and are tested on the held-out 120
plus 120, the same split at every layer: a logistic
regression with strong regularization (C=0.01) on
standardized features, and the same probe after a PCA
reduction to 64 components. Each helper is shadowed by a
shuffled-label control: 5 shuffles, each trained and
tested identically, reported as their mean and standard
deviation. Results land in `property_probe_results.csv`:
one row per layer per task per readout, controls included.

**The attention readout.** The first 6 tone and first 6
silence clips get a second pass with attentions returned.
For every layer, the mean attention from the final
sequence position to the verified audio positions lands
in `property_attention.csv`.

**The tail.** The runner ends with a plain-language
readout: for each task and readout, the first layer where
the helper falls below 0.70 and stays below (or "never"),
and the shuffled controls' range across the whole run.

In [ ]:
# --- Battery mode: the tracing round 2 battery -----------------------------------------
# Same runner, same session Python, streamed live. The runner
# saves its outputs itself, to the Drive tracing folder and
# to /content. A nonzero exit code means the token check
# failed, a feed hard-stopped the run, clip generation failed
# its measurements, or the extraction count mismatched (the
# banners above say which).
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_property_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: the token check failed, a feed proof hard-stopped")
    print("the run, clip generation failed, or the counts mismatched. Show")
    print("it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: token check passed, features extracted and counted,")
print("helpers trained with 5-shuffle controls, attention read, CSVs saved.")

## Closing · How to read the answers

Read each task's curve on its own first: one accuracy per
layer per readout, from the embedding output (layer 0) to
the last thinker layer, with the shuffled control's mean
beside it. The token check transcript,
`property_token_check.txt`, is read before any curve: it
is the proof of which positions every number pooled over,
and how many there were.

- **The tone fades somewhere.** The fade layer is the
  neighborhood where this helper stops reading the tone.
  Read it with the attention numbers beside it: if the
  final position's attention to the tone's positions was
  near zero all along, the answer never looked; if
  attention was healthy and the report still missed, it
  looked and dropped. Either way, the fade is a
  readability result, not yet a proof of loss. The surgery
  build is the causal check that confirms or kills the
  claim.
- **The speech reads at every layer.** Then nothing about
  the speech is lost inside this model, and its denial in
  the outside tests lives in how the presence answer is
  formed, past the information. That was the registered
  prediction.
- **The speech fades somewhere.** Then the prediction dies
  in public, the taxonomy changes, and the write-up says so
  plainly. A result that kills a prediction is a result.
- **A control beats chance.** The run is void until fixed,
  under the plan's rules, whatever the curves look like.
  With 240 training clips per class, two regularized
  readouts, and the control reported as the mean of 5
  shuffles beside every score, round 2 is built so this
  verdict, whichever way it lands, can be trusted.
- **Scope.** Every claim stays capped at this model, these
  sounds, this question per class, this run. Pegasus stays
  a black box; MiniCPM and OLA tracing wait until these
  instruments have run end to end here; the picture
  condition is a later round.

When you finish, the files in the `tracing-qwen3b` Drive
folder are the record: the token check transcript, the
manifest, the features, the probe results, and the
attention readout. Send them back, and the tracing gets
written up next to the codec control.